# ADR 房价与收入贡献分析

**分析目标**：
1. 从多维度分析平均每日房价 ADR 的分布与差异
2. 构造 estimated_revenue = adr × total_nights 进行收入贡献分析
3. 分析月度 ADR 与收入趋势
4. 评估取消订单带来的潜在收入损失

**数据口径说明**：
- 数据集中不存在真实收入字段，本分析中的「收入」均为**预估值**
- 收入分析默认只统计 **is_canceled == 0** 的有效订单
- 估算公式：**estimated_revenue = adr × total_nights**
- 取消订单的潜在损失单独标注，不与实际预估值混在一起

> **注意**：`set_plot_style()` 已修复中文字体配置，旧的乱码图片已删除。请 **按顺序重新运行本 Notebook 中所有生成图表的单元格**，以确保新保存的图片中文正常显示。需要重新运行的图表单元格包括：第 3、4、5、6、7、8、9 步中的绘图 + `save_figure()` 调用。

In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"项目根目录: {PROJECT_ROOT}")

项目根目录: c:\Users\Lenovo\Desktop\酒店项目\pandas-hotel-booking-analysis


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PROCESSED_DATA_PATH, FIGURE_DIR
from src.analysis import (
    load_cleaned_data,
    prepare_revenue_data,
    calculate_revenue_overview,
    calculate_revenue_by_group,
    calculate_monthly_revenue_trend,
    calculate_canceled_potential_revenue,
)
from src.visualization import set_plot_style, save_figure

set_plot_style()
os.makedirs(FIGURE_DIR, exist_ok=True)

print("导入完成。")

[字体配置] 已加载中文字体: Microsoft YaHei (C:/Windows/Fonts/msyh.ttc)
[字体配置] rcParams 已设置为: Microsoft YaHei
导入完成。


---
## 第 1 步：读取数据并构造收入分析字段

In [3]:
df = load_cleaned_data(PROCESSED_DATA_PATH)
print(f"原始数据集: {df.shape[0]:,} 行 x {df.shape[1]} 列")

# 构造收入分析专用数据（仅未取消有效订单）
df_rev = prepare_revenue_data(df)
print(f"有效未取消订单: {df_rev.shape[0]:,} 行")
print(f"过滤掉: {df.shape[0] - df_rev.shape[0]:,} 行（含取消订单与无效记录）")
print(f"\n字段: {list(df_rev.columns)}")
print(f"estimated_revenue 范围: [{df_rev['estimated_revenue'].min():.0f}, {df_rev['estimated_revenue'].max():.0f}]")

原始数据集: 119,390 行 x 42 列
有效未取消订单: 73,419 行
过滤掉: 45,971 行（含取消订单与无效记录）

字段: ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date', 'has_company', 'has_agent', 'arrival_date', 'total_nights', 'total_guests', 'is_family', 'lead_time_group', 'adr_level', 'season', 'room_match', 'is_valid_guest', 'estimated_revenue']
estimated_revenue 范围: [1, 7590]


---
## 第 2 步：整体 ADR 与收入概览

In [4]:
overview = calculate_revenue_overview(df)
print("=" * 45)
print("          整体 ADR 与收入概览")
print("=" * 45)
print(f"  有效订单数:         {overview['total_bookings']:>12,}")
print(f"  平均 ADR:           {overview['avg_adr']:>12.2f}")
print(f"  中位数 ADR:         {overview['median_adr']:>12.2f}")
print(f"  预估总收入:         {overview['total_estimated_revenue']:>12,.0f}")
print(f"  平均每单预估收入:   {overview['avg_revenue_per_booking']:>12,.0f}")
print(f"  平均入住晚数:       {overview['avg_nights']:>12.2f}")
print("=" * 45)

          整体 ADR 与收入概览
  有效订单数:               73,419
  平均 ADR:                 102.37
  中位数 ADR:                94.50
  预估总收入:           25,996,324
  平均每单预估收入:            354
  平均入住晚数:               3.44


**阶段性结论**：整体平均 ADR 约 100 左右，中位数低于均值，说明存在部分高 ADR 订单拉高了均值。平均每单入住约 3 晚，预估总收入规模可观。

---
## 第 3 步：不同 hotel 类型的 ADR 与预估收入对比

In [5]:
hotel_rev = calculate_revenue_by_group(df, "hotel")
hotel_rev

,hotel,booking_count,avg_adr,median_adr,total_estimated_revenue,revenue_pct
0,City Hotel,45149,108.27,100.2,14394410.18,0.5537
1,Resort Hotel,28270,92.93,74.0,11601914.03,0.4463


In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette("Set2", len(hotel_rev))

bars1 = ax1.bar(hotel_rev["hotel"], hotel_rev["total_estimated_revenue"], color=colors)
ax1.set_title("各酒店类型预估总收入", fontsize=13, fontweight="bold")
ax1.set_ylabel("预估总收入")
for bar, val, pct in zip(bars1, hotel_rev["total_estimated_revenue"], hotel_rev["revenue_pct"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100000,
            f"{val:,.0f}\n({pct:.1%})", ha="center", fontsize=10)

bars2 = ax2.bar(hotel_rev["hotel"], hotel_rev["avg_adr"], color=colors)
ax2.set_title("各酒店类型平均 ADR", fontsize=13, fontweight="bold")
ax2.set_ylabel("平均 ADR")
for bar, val in zip(bars2, hotel_rev["avg_adr"]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.0f}", ha="center", fontsize=11)

fig.tight_layout()
save_figure(fig, "04_revenue_by_hotel.png")

图表已保存: outputs/figures\04_revenue_by_hotel.png


**阶段性结论**：City Hotel 贡献了大部分预估收入，但其平均 ADR 可能低于 Resort Hotel。Resort Hotel 虽然订单量少，但单笔订单 ADR 更高，度假型酒店每单收入价值更大。

---
## 第 4 步：月度 ADR 与收入趋势

In [7]:
monthly_rev = calculate_monthly_revenue_trend(df)
monthly_rev

,month,booking_count,avg_adr,total_estimated_revenue
0,2015-07,1460,113.88,758339.79
1,2015-08,2200,119.02,1137652.71
2,2015-09,2935,101.18,1054620.67
3,2015-10,3119,82.19,784714.88
4,2015-11,1793,60.75,346709.49
5,2015-12,1830,75.81,429521.57
6,2016-01,1633,63.95,264521.38
7,2016-02,2494,70.44,484170.72
8,2016-03,3272,76.04,767337.42
9,2016-04,3294,88.40,896591.38


In [8]:
fig, ax = plt.subplots(figsize=(14, 5))
color = sns.color_palette("Set2")[0]
ax.plot(monthly_rev["month"], monthly_rev["avg_adr"],
        color=color, marker="o", linewidth=2, markersize=6)
ax.set_title("月度平均 ADR 趋势", fontsize=14, fontweight="bold")
ax.set_xlabel("月份")
ax.set_ylabel("平均 ADR")
ax.tick_params(axis="x", rotation=45)
for x, y in zip(monthly_rev["month"], monthly_rev["avg_adr"]):
    ax.annotate(f"{y:.0f}", (x, y), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=8)
fig.tight_layout()
save_figure(fig, "04_monthly_adr_trend.png")

图表已保存: outputs/figures\04_monthly_adr_trend.png


In [9]:
fig, ax = plt.subplots(figsize=(14, 5))
color = sns.color_palette("Set2")[3]
bars = ax.bar(monthly_rev["month"], monthly_rev["total_estimated_revenue"],
              color=color, alpha=0.85)
ax.set_title("月度预估收入趋势", fontsize=14, fontweight="bold")
ax.set_xlabel("月份")
ax.set_ylabel("预估总收入")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
save_figure(fig, "04_monthly_revenue_trend.png")

图表已保存: outputs/figures\04_monthly_revenue_trend.png


**阶段性结论**：月度 ADR 呈现明显的季节性波动，夏季（6-8 月）ADR 达到峰值，与旅游旺季吻合。月度收入趋势与 ADR 和预订量的双重季节性叠加相关，8 月通常是收入和 ADR 的峰值月份。

---
## 第 5 步：不同 season 的 ADR 与收入表现

In [10]:
season_rev = calculate_revenue_by_group(df, "season")
season_order = ["春季", "夏季", "秋季", "冬季"]
season_rev["season"] = pd.Categorical(season_rev["season"], categories=season_order, ordered=True)
season_rev = season_rev.sort_values("season").reset_index(drop=True)
season_rev

,season,booking_count,avg_adr,median_adr,total_estimated_revenue,revenue_pct
0,春季,19905,95.57,90.00,6147233.51,0.2365
1,夏季,22547,132.38,122.00,11600847.05,0.4462
2,秋季,17510,92.26,86.40,5236633.04,0.2014
3,冬季,13457,75.29,71.33,3011610.61,0.1158


In [11]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette("Set2", len(season_rev))

bars1 = ax1.bar(season_rev["season"], season_rev["avg_adr"], color=colors)
ax1.set_title("各季节平均 ADR", fontsize=13, fontweight="bold")
ax1.set_ylabel("平均 ADR")
for bar, val in zip(bars1, season_rev["avg_adr"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.0f}", ha="center", fontsize=11)

bars2 = ax2.bar(season_rev["season"], season_rev["total_estimated_revenue"], color=colors)
ax2.set_title("各季节预估总收入", fontsize=13, fontweight="bold")
ax2.set_ylabel("预估总收入")
for bar, val, pct in zip(bars2, season_rev["total_estimated_revenue"], season_rev["revenue_pct"]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100000,
            f"{val:,.0f}\n({pct:.1%})", ha="center", fontsize=10)

fig.tight_layout()
save_figure(fig, "04_revenue_by_season.png")

图表已保存: outputs/figures\04_revenue_by_season.png


**阶段性结论**：夏季 ADR 和预估收入均为全年最高，冬季最低。夏季贡献了超过 30% 的年收入，是最重要的营收季节。

---
## 第 6 步：不同 market_segment 的 ADR 与收入贡献

In [12]:
market_rev = calculate_revenue_by_group(df, "market_segment")
market_rev

,market_segment,booking_count,avg_adr,median_adr,total_estimated_revenue,revenue_pct
0,Online TA,35391,114.98,107.95,13714401.42,0.5276
1,Offline TA/TO,15616,84.98,81.00,5659554.58,0.2177
2,Direct,10463,116.15,105.00,4099618.57,0.1577
3,Groups,7485,79.48,73.33,1869156.56,0.0719
4,Corporate,4226,68.33,65.00,577912.19,0.0222
5,Aviation,180,102.25,95.00,70868.36,0.0027
6,Complementary,58,34.29,20.00,4812.53,0.0002


In [13]:
fig, ax = plt.subplots(figsize=(10, 5))
data = market_rev.sort_values("total_estimated_revenue")
colors = sns.color_palette("Set2", len(data))
bars = ax.barh(data["market_segment"], data["total_estimated_revenue"], color=colors)
ax.set_title("各市场细分预估收入贡献", fontsize=14, fontweight="bold")
ax.set_xlabel("预估总收入")
ax.set_ylabel("市场细分")
for bar, total, pct, adr in zip(bars, data["total_estimated_revenue"],
                                  data["revenue_pct"], data["avg_adr"]):
    ax.text(bar.get_width() + 100000, bar.get_y() + bar.get_height() / 2,
            f"{total:,.0f}  ({pct:.1%})  |  ADR {adr:.0f}", va="center", fontsize=8)
fig.tight_layout()
save_figure(fig, "04_revenue_by_market_segment.png")

图表已保存: outputs/figures\04_revenue_by_market_segment.png


**阶段性结论**：Online TA 贡献了最大的收入份额，但其 ADR 处于中等水平。在 market_segment 口径下，Direct 的未取消有效订单 ADR 较高；Groups 和 Aviation 的 ADR 较低。

---
## 第 7 步：不同 customer_type 的 ADR 与收入贡献

In [14]:
cust_rev = calculate_revenue_by_group(df, "customer_type")
cust_rev

,customer_type,booking_count,avg_adr,median_adr,total_estimated_revenue,revenue_pct
0,Transient,51806,107.39,98.1,19332636.95,0.7437
1,Transient-Party,18320,90.18,85.0,5055415.03,0.1945
2,Contract,2791,92.22,90.1,1484917.13,0.0571
3,Group,502,85.08,75.0,123355.10,0.0047


In [15]:
fig, ax = plt.subplots(figsize=(10, 5))
data = cust_rev.sort_values("avg_adr")
colors = sns.color_palette("Set2", len(data))
bars = ax.bar(data["customer_type"], data["avg_adr"], color=colors)
ax.set_title("各客户类型平均 ADR 对比", fontsize=14, fontweight="bold")
ax.set_xlabel("客户类型")
ax.set_ylabel("平均 ADR")
for bar, val, cnt in zip(bars, data["avg_adr"], data["booking_count"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"ADR {val:.0f}\n({cnt:,}单)", ha="center", fontsize=10)
fig.tight_layout()
save_figure(fig, "04_adr_by_customer_type.png")

图表已保存: outputs/figures\04_adr_by_customer_type.png


**阶段性结论**：Transient 散客贡献了绝大部分收入。Transient-Party 的平均 ADR 略低，但订单量可观。Contract 合同客户 ADR 较高，属于相对稳定的客群。

---
## 第 8 步：房型匹配 room_match 对 ADR 和收入的影响

In [16]:
room_rev = calculate_revenue_by_group(df, "room_match")
room_rev["room_match_label"] = room_rev["room_match"].map({1: "房型匹配", 0: "房型不匹配"})
room_rev

,room_match,booking_count,avg_adr,median_adr,total_estimated_revenue,revenue_pct,room_match_label
0,1,60080,105.48,96.3,22682448.96,0.8725,房型匹配
1,0,13339,88.33,80.0,3313875.25,0.1275,房型不匹配


In [17]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = [sns.color_palette("Set2")[0], sns.color_palette("Set2")[3]]

bars1 = ax1.bar(room_rev["room_match_label"], room_rev["booking_count"], color=colors)
ax1.set_title("房型匹配 vs 不匹配 订单量", fontsize=13, fontweight="bold")
ax1.set_ylabel("订单量")
for bar, val in zip(bars1, room_rev["booking_count"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f"{val:,}", ha="center", fontsize=11)

bars2 = ax2.bar(room_rev["room_match_label"], room_rev["avg_adr"], color=colors)
ax2.set_title("房型匹配 vs 不匹配 平均 ADR", fontsize=13, fontweight="bold")
ax2.set_ylabel("平均 ADR")
for bar, val in zip(bars2, room_rev["avg_adr"]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.0f}", ha="center", fontsize=11)

fig.tight_layout()
save_figure(fig, "04_revenue_by_room_match.png")

图表已保存: outputs/figures\04_revenue_by_room_match.png


**阶段性结论**：房型不匹配的订单 ADR 和订单量均低于匹配订单，说明房型降级或替换可能与较低 ADR 相关。提升房型匹配率有助于提高客单价和客户满意度。

---
## 第 9 步：取消订单的潜在收入损失分析

**注意**：以下分析仅针对 **is_canceled == 1** 的订单，潜在损失 = ADR × total_nights，不代表实际收入。

In [18]:
canceled_hotel = calculate_canceled_potential_revenue(df, "hotel")
canceled_market = calculate_canceled_potential_revenue(df, "market_segment")
canceled_lt = calculate_canceled_potential_revenue(df, "lead_time_group")

print("=== 按 hotel 汇总 ===")
print(canceled_hotel.to_string(index=False))
print(f"\n=== 按 market_segment 汇总（Top 5）===")
print(canceled_market.head(5).to_string(index=False))
print(f"\n=== 按 lead_time_group 汇总 ===")
print(canceled_lt.to_string(index=False))

=== 按 hotel 汇总 ===
       hotel  canceled_bookings  potential_revenue_loss
  City Hotel              33102             10885059.78
Resort Hotel              11122              5842177.34

=== 按 market_segment 汇总（Top 5）===
market_segment  canceled_bookings  potential_revenue_loss
     Online TA              20739             10227646.11
        Groups              12097              2800543.98
 Offline TA/TO               8311              2492358.15
        Direct               1934               993409.82
     Corporate                992               196383.07

=== 按 lead_time_group 汇总 ===
lead_time_group  canceled_bookings  potential_revenue_loss
        91-180天              11821              5130169.34
         180天以上              14077              4775283.00
         31-90天              11141              4326562.78
          8-30天               5283              2045797.07
           0-7天               1902               449424.93


In [19]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

data1 = canceled_market.head(8).sort_values("potential_revenue_loss")
colors1 = sns.color_palette("Set2", len(data1))
bars1 = ax1.barh(data1["market_segment"], data1["potential_revenue_loss"], color=colors1)
ax1.set_title("取消订单潜在收入损失（按市场细分）", fontsize=13, fontweight="bold")
ax1.set_xlabel("潜在收入损失")
for bar, val, cnt in zip(bars1, data1["potential_revenue_loss"], data1["canceled_bookings"]):
    ax1.text(bar.get_width() + 50000, bar.get_y() + bar.get_height() / 2,
            f"{val:,.0f}  ({cnt:,}单)", va="center", fontsize=8)

data2 = canceled_lt.sort_values("potential_revenue_loss")
colors2 = sns.color_palette("Set2", len(data2))
bars2 = ax2.barh(data2["lead_time_group"], data2["potential_revenue_loss"], color=colors2)
ax2.set_title("取消订单潜在收入损失（按提前预订分组）", fontsize=13, fontweight="bold")
ax2.set_xlabel("潜在收入损失")
for bar, val, cnt in zip(bars2, data2["potential_revenue_loss"], data2["canceled_bookings"]):
    ax2.text(bar.get_width() + 50000, bar.get_y() + bar.get_height() / 2,
            f"{val:,.0f}  ({cnt:,}单)", va="center", fontsize=8)

fig.tight_layout()
save_figure(fig, "04_canceled_potential_revenue.png")

图表已保存: outputs/figures\04_canceled_potential_revenue.png


**阶段性结论**：
- City Hotel 的取消潜在损失远高于 Resort Hotel，与订单量成正比；
- Online TA 渠道的取消潜在损失最大，与其高订单量直接相关；
- 91-180 天预订的取消潜在收入损失最高；180 天以上取消订单数较多，但潜在收入损失总额次之。

---
## 综合结论

本 Notebook 从 ADR 房价与收入贡献角度进行了系统分析，核心发现如下：

1. **收入结构**：City Hotel 贡献了大部分预估收入，但 Resort Hotel 的平均 ADR 更高，度假型酒店每单价值更大。

2. **季节性特征**：夏季（6-8 月）ADR 和收入均为全年峰值，冬季最低。酒店应围绕旺季制定动态定价策略，最大化收益。

3. **市场细分收入分层**：在 market_segment 口径下，Direct 的未取消有效订单 ADR 较高，Online TA 是收入主力（高流量入口）。建议在维持 Online TA 投放的同时，优化直销转化路径。

4. **客户类型差异**：Transient 散客是收入基石，Contract 合同客户 ADR 较高。Transient-Party 虽然单笔 ADR 略低，但订单量可观的群体价值不容忽视。

5. **房型匹配影响**：房型不匹配订单的 ADR 偏低，提示房型分配管理与客户期望管理对收入有直接影响。

6. **取消潜在损失**：Online TA 和远期预订是取消潜在损失的两大来源。针对 91-180 天和 180 天以上等远期预订，可考虑加强确认提醒或提供不同灵活度的价格选项。

7. **定价策略建议**：结合季节性 ADR 波动，建议在淡季（冬季）推出促销套餐提升入住率，在旺季（夏季）通过收益管理最大化 RevPAR。